# HyDE: reproduction on SciFact and ArguAna

The claim we test (Gao et al., ACL 2023, Table 2): searching with the embedding of LLM-written hypothetical documents beats searching with the query embedding, with the same unsupervised encoder (Contriever) and no training.

Paper, nDCG@10: SciFact 64.9 with Contriever and 69.1 with HyDE; ArguAna 37.9 and 46.6.

Same as the paper: `facebook/contriever`, mean pooling, dot product, full documents, the paper's prompts, N = 8, temperature 0.7, at most 512 tokens, the query vector included in the average.
Different: the generator. InstructGPT (`text-davinci-003`) is retired; we use Qwen2.5-3B-Instruct, 8-bit, on a laptop (`generate.py`).

Run from the repo root after `generate.py`, or with the committed `generations/`. Scores come from `testbed.score_run` (`pytrec_eval`; on ArguAna the query's own document is removed).

In [1]:
import sys
sys.path += ["../..", "."]
from sentence_transformers import SentenceTransformer
import testbed, hyde

model = SentenceTransformer("facebook/contriever", device=testbed.device())   # mean pooling
model.max_seq_length = 512

## Plain Contriever, then HyDE

For each dataset: the plain Contriever score (the check against the paper), then HyDE for N = 1, 2, 4, 8 hypothetical documents, with and without the query vector in the average. Every HyDE line is compared with plain Contriever on the same queries: mean difference, bootstrap 95 % CI, paired t-test.

In [2]:
for name in ["scifact", "arguana"]:
    corpus, queries, qrels = testbed.load(name)
    ids, qids = list(corpus), list(queries)
    docs = [(d["title"] + " " + d["text"]).strip() for d in corpus.values()]    # full text, no 200-word cut
    D = model.encode(docs, batch_size=32, convert_to_numpy=True)
    Q = model.encode(list(queries.values()), batch_size=32, convert_to_numpy=True)
    E = hyde.embed_generations(model, queries, hyde.load_generations(name))
    _, base, base_r = testbed.score_run(hyde.search(Q, D, qids, ids), name)
    print(f"{name} Contriever            nDCG@10 {base.mean():.3f}  R@100 {base_r.mean():.3f}")
    for n in (1, 2, 4, 8):
        for label, q in (("with query", Q), ("docs only ", None)):
            _, nd, rc = testbed.score_run(hyde.search(hyde.hyde_vectors(E, q, n), D, qids, ids), name)
            c = testbed.compare(nd, base)
            print(f"{name} HyDE N={n} {label}  nDCG@10 {nd.mean():.3f}  R@100 {rc.mean():.3f}  "
                  f"delta {c['delta']:+.3f} {c['ci95']} p={c['p']:.3g}")

scifact Contriever            nDCG@10 0.649  R@100 0.926
scifact HyDE N=1 with query  nDCG@10 0.651  R@100 0.944  delta +0.001 [-0.0268, 0.0294] p=0.928


scifact HyDE N=1 docs only   nDCG@10 0.544  R@100 0.890  delta -0.105 [-0.1451, -0.0643] p=4.91e-07
scifact HyDE N=2 with query  nDCG@10 0.636  R@100 0.944  delta -0.013 [-0.0443, 0.0186] p=0.419


scifact HyDE N=2 docs only   nDCG@10 0.558  R@100 0.896  delta -0.092 [-0.1302, -0.053] p=4.88e-06
scifact HyDE N=4 with query  nDCG@10 0.617  R@100 0.930  delta -0.032 [-0.0655, 0.0001] p=0.0523


scifact HyDE N=4 docs only   nDCG@10 0.574  R@100 0.918  delta -0.076 [-0.1136, -0.0376] p=0.000117
scifact HyDE N=8 with query  nDCG@10 0.608  R@100 0.928  delta -0.042 [-0.0765, -0.0073] p=0.0193


scifact HyDE N=8 docs only   nDCG@10 0.584  R@100 0.918  delta -0.065 [-0.1028, -0.0275] p=0.000769


arguana Contriever            nDCG@10 0.379  R@100 0.901


... (4 outputs left out)

arguana HyDE N=4 with query  nDCG@10 0.223  R@100 0.818  delta -0.157 [-0.173, -0.1403] p=1.71e-69


arguana HyDE N=4 docs only   nDCG@10 0.142  R@100 0.690  delta -0.237 [-0.2561, -0.219] p=3.45e-115


arguana HyDE N=8 with query  nDCG@10 0.192  R@100 0.773  delta -0.187 [-0.2046, -0.1697] p=1.76e-84


arguana HyDE N=8 docs only   nDCG@10 0.147  R@100 0.692  delta -0.232 [-0.251, -0.2139] p=1.41e-110
